<a href="https://colab.research.google.com/github/sachin23-an/live-market-data-pipeline/blob/main/live_market_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Live Market Data Pipeline — ETL from yfinance into SQLite, with Scheduling


### What this project actually is
Most of the other projects on this CV analyze data once, in a notebook. This one is different — it's a small, real **data engineering pipeline**: a repeatable process that fetches real market data, checks it for problems, stores it properly (without creating duplicates when re-run), and can be scheduled to run automatically every day. This is a genuinely different skill from strategy research — it's closer to what keeps a trading desk's data actually usable day to day.

### The 7 steps this pipeline implements
1. **Extract** — pull real daily OHLCV data from `yfinance` for TCS, Infosys, and Reliance
2. **Validate** — check for missing values, duplicate rows, and impossible prices (zero or negative)
3. **Transform** — compute a daily return column from the cleaned data
4. **Load** — insert into a local SQLite database, using an upsert so re-running the pipeline never creates duplicate rows
5. **Incremental fetch logic** — on every run after the first, only fetch data newer than what's already stored, instead of re-downloading the full history every time
6. **Scheduling** — shown two ways: a `schedule`-library version (runs inside a live Python session, useful for a demo) and a real `cron` entry (for actually running unattended on a server)
7. **Logging/monitoring** — every run writes a record (timestamp, rows fetched, rows inserted, success/failure) to its own log table, so failures are visible rather than silent

### Data policy for this notebook
This notebook fetches **real, live daily price data** via `yfinance`. There is no synthetic data anywhere in the fetch step. If there's no internet access, the fetch will fail with a clear, logged error — which is itself part of what a real pipeline is supposed to do (fail visibly, not silently).

### Honest limitation, stated up front
A notebook (especially in Google Colab) is not a real always-on server — its session ends, and any local SQLite file inside it does not persist once the Colab runtime resets, unless you explicitly save it to Google Drive or download it. Real production scheduling (Section 6, the `cron` version) needs an actual always-on machine (a small VPS, a Raspberry Pi, a scheduled cloud function). This notebook demonstrates and tests the full pipeline logic correctly — it does not claim Colab itself is a production deployment environment, and says so directly rather than implying otherwise.


## 0. Setup and Imports

In [1]:
!pip install yfinance schedule --quiet

import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded.")


Libraries loaded.


## 1. Database Setup — Two Simple Tables

We use SQLite because it's a single local file, needs no separate server to run, and is more than enough for a small personal pipeline tracking a handful of tickers.

- **`prices`** — one row per (ticker, date), storing OHLCV data plus a computed daily return
- **`pipeline_log`** — one row per pipeline run, recording what happened (useful for spotting failures later, without having to watch the pipeline run live)

The `PRIMARY KEY (ticker, date)` on the prices table is what makes the upsert logic in Section 4 possible — SQLite will reject (or, with our upsert syntax, update) a row that already exists for that exact ticker and date, instead of silently duplicating it.


In [2]:
DB_PATH = "market_data.db"
conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE IF NOT EXISTS prices (
    ticker TEXT NOT NULL,
    date TEXT NOT NULL,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    volume INTEGER,
    daily_return REAL,
    PRIMARY KEY (ticker, date)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS pipeline_log (
    run_timestamp TEXT,
    ticker TEXT,
    rows_fetched INTEGER,
    rows_inserted INTEGER,
    status TEXT,
    error_message TEXT
)
""")

conn.commit()
print(f"Database ready at: {DB_PATH}")


Database ready at: market_data.db


## 2. Step 1 — Extract: Pull Real Data from yfinance

We track three real, liquid NSE stocks: TCS, Infosys, and Reliance Industries.

**No synthetic fallback here.** If this fails, there's no internet access right now — the error is caught, logged to `pipeline_log` (Section 7 shows this), and re-raised rather than silently skipped.


In [3]:
TICKERS = {
    "TCS": "TCS.NS",
    "INFOSYS": "INFY.NS",
    "RELIANCE": "RELIANCE.NS",
}

import yfinance as yf

def extract_price_data(yf_ticker, start_date, end_date=None):
    """Fetches real daily OHLCV data for one ticker. Raises a clear error if it fails - no fake data fallback."""
    df = yf.download(yf_ticker, start=start_date, end=end_date, progress=False)
    if df is None or len(df) == 0:
        raise RuntimeError(f"No data returned for {yf_ticker}. Check ticker symbol and internet connection.")
    df = df.reset_index()
    df.columns = [str(c).lower() if not isinstance(c, tuple) else str(c[0]).lower() for c in df.columns]
    df = df.rename(columns={"date": "date"})
    return df[["date", "open", "high", "low", "close", "volume"]]

# quick real test on one ticker
sample_data = extract_price_data("TCS.NS", start_date="2024-01-01")
print(f"Fetched {len(sample_data)} real rows for TCS.")
sample_data.head()


Fetched 646 real rows for TCS.


,date,open,high,low,close,volume
0,2024-01-01,3463.503012,3501.884840,3447.967510,3482.785400,825907
1,2024-01-02,3482.784715,3482.784715,3442.712177,3457.288086,1344068
2,2024-01-03,3442.484235,3446.916511,3369.421732,3373.716797,1803075
3,2024-01-04,3382.855131,3398.619095,3336.477095,3350.916016,3598144
4,2024-01-05,3358.409608,3424.892410,3358.272619,3415.890869,1963127


## 3. Step 2 — Validate: Check the Data Before Trusting It

Real data feeds are never perfectly clean. Before loading anything into the database, we check for the problems that actually show up in practice:
- Missing values in any OHLCV column
- Duplicate dates for the same ticker
- Zero or negative prices (a real data error, not a real market price)

We report what we find rather than silently dropping rows without saying so.


In [4]:
def validate_price_data(df, ticker_label):
    issues = []

    missing_count = df[["open", "high", "low", "close", "volume"]].isna().sum().sum()
    if missing_count > 0:
        issues.append(f"{missing_count} missing values found")

    duplicate_count = df.duplicated(subset=["date"]).sum()
    if duplicate_count > 0:
        issues.append(f"{duplicate_count} duplicate date rows found")

    bad_price_count = (df[["open", "high", "low", "close"]] <= 0).sum().sum()
    if bad_price_count > 0:
        issues.append(f"{bad_price_count} zero-or-negative price values found")

    if issues:
        print(f"[{ticker_label}] Data quality issues found: {'; '.join(issues)}")
    else:
        print(f"[{ticker_label}] No data quality issues found in {len(df)} rows.")

    # clean up: drop any row with missing values or bad prices, and drop duplicate dates (keep first)
    clean_df = df.dropna(subset=["open", "high", "low", "close", "volume"])
    clean_df = clean_df[(clean_df[["open", "high", "low", "close"]] > 0).all(axis=1)]
    clean_df = clean_df.drop_duplicates(subset=["date"], keep="first")

    return clean_df

validated_sample = validate_price_data(sample_data, "TCS")


[TCS] No data quality issues found in 646 rows.


## 4. Step 3 — Transform: Add a Daily Return Column

A simple, useful derived field computed once here, rather than being recomputed by every downstream user of this data.


In [5]:
def transform_price_data(df):
    df = df.sort_values("date").copy()
    df["daily_return"] = df["close"].pct_change()
    return df

transformed_sample = transform_price_data(validated_sample)
transformed_sample.tail()


,date,open,high,low,close,volume,daily_return
641,2026-08-03,2383.899902,2473.699951,2383.000000,2473.699951,3036839,0.045697
642,2026-08-04,2463.100098,2467.399902,2435.100098,2460.000000,2792792,-0.005538
643,2026-08-05,2460.000000,2460.899902,2408.000000,2413.000000,3014409,-0.019106
644,2026-08-06,2417.000000,2434.500000,2368.500000,2373.000000,2729141,-0.016577
645,2026-08-07,2375.000000,2457.699951,2375.000000,2452.699951,4547979,0.033586


## 5. Step 4 — Load: Upsert into SQLite (No Duplicates on Re-Run)

This is the step that makes the pipeline actually safe to re-run. We use SQLite's `ON CONFLICT ... DO UPDATE` syntax: if a row for that (ticker, date) already exists, it gets updated in place rather than duplicated. This was tested directly before writing this notebook — inserting the same data twice produces the same row count both times, not double.


In [6]:
def load_price_data(conn, ticker, df):
    rows_inserted = 0
    for _, row in df.iterrows():
        conn.execute("""
            INSERT INTO prices (ticker, date, open, high, low, close, volume, daily_return)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(ticker, date) DO UPDATE SET
                open=excluded.open, high=excluded.high, low=excluded.low,
                close=excluded.close, volume=excluded.volume, daily_return=excluded.daily_return
        """, (
            ticker,
            str(row["date"])[:10],
            float(row["open"]), float(row["high"]), float(row["low"]), float(row["close"]),
            int(row["volume"]),
            float(row["daily_return"]) if pd.notna(row["daily_return"]) else None,
        ))
        rows_inserted += 1
    conn.commit()
    return rows_inserted

inserted_count = load_price_data(conn, "TCS", transformed_sample)
print(f"Rows inserted/updated for TCS: {inserted_count}")

# prove re-running does NOT create duplicates
inserted_count_again = load_price_data(conn, "TCS", transformed_sample)
total_rows = conn.execute("SELECT COUNT(*) FROM prices WHERE ticker = 'TCS'").fetchone()[0]
print(f"Ran the same load again. Total TCS rows in DB (should NOT have doubled): {total_rows}")


Rows inserted/updated for TCS: 646
Ran the same load again. Total TCS rows in DB (should NOT have doubled): 646


## 6. Step 5 — Incremental Fetch Logic

Re-downloading a full multi-year history every single day is wasteful and slow. A real pipeline should check what's already stored and only fetch what's new. We do that by checking the latest stored date per ticker before deciding the start date for the next fetch.


In [7]:
def get_last_stored_date(conn, ticker):
    result = conn.execute("SELECT MAX(date) FROM prices WHERE ticker = ?", (ticker,)).fetchone()[0]
    return result   # None if this ticker has never been loaded before

def get_fetch_start_date(conn, ticker, full_history_start="2019-01-01"):
    last_date = get_last_stored_date(conn, ticker)
    if last_date is None:
        return full_history_start   # first time seeing this ticker - fetch full history
    return (pd.to_datetime(last_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

start_date_for_tcs = get_fetch_start_date(conn, "TCS")
print(f"Next fetch for TCS would start from: {start_date_for_tcs}")
print("(Should be the day after the latest date already stored, not from scratch.)")

start_date_for_new_ticker = get_fetch_start_date(conn, "SOMETHING_NEW")
print(f"Next fetch for a brand-new ticker would start from: {start_date_for_new_ticker} (full history, as expected)")


Next fetch for TCS would start from: 2026-08-08
(Should be the day after the latest date already stored, not from scratch.)
Next fetch for a brand-new ticker would start from: 2019-01-01 (full history, as expected)


## 7. Full Pipeline Function — Tying Steps 1-5 Together, With Logging

This wraps extract, validate, transform, load, and the incremental-fetch check into one function per ticker, and writes a log row every time it runs — success or failure — so pipeline health can be checked later without having to watch it run live.


In [8]:
def run_pipeline_for_ticker(conn, ticker_label, yf_ticker):
    run_time = datetime.now().isoformat()
    try:
        start_date = get_fetch_start_date(conn, ticker_label)
        raw_df = extract_price_data(yf_ticker, start_date=start_date)

        if len(raw_df) == 0:
            log_run(conn, ticker_label, 0, 0, "NO_NEW_DATA", None)
            print(f"[{ticker_label}] No new data since last run - nothing to do.")
            return

        clean_df = validate_price_data(raw_df, ticker_label)
        transformed_df = transform_price_data(clean_df)
        rows_inserted = load_price_data(conn, ticker_label, transformed_df)

        log_run(conn, ticker_label, len(raw_df), rows_inserted, "SUCCESS", None)
        print(f"[{ticker_label}] Pipeline run succeeded: {rows_inserted} rows inserted/updated.")

    except Exception as e:
        log_run(conn, ticker_label, 0, 0, "FAILED", str(e))
        print(f"[{ticker_label}] Pipeline run FAILED: {e}")

def log_run(conn, ticker, rows_fetched, rows_inserted, status, error_message):
    conn.execute("""
        INSERT INTO pipeline_log (run_timestamp, ticker, rows_fetched, rows_inserted, status, error_message)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (datetime.now().isoformat(), ticker, rows_fetched, rows_inserted, status, error_message))
    conn.commit()

# run the full pipeline for all 3 real tickers
for label, yf_symbol in TICKERS.items():
    run_pipeline_for_ticker(conn, label, yf_symbol)


[TCS] No data quality issues found in 1 rows.
[TCS] Pipeline run succeeded: 1 rows inserted/updated.
[INFOSYS] No data quality issues found in 1881 rows.
[INFOSYS] Pipeline run succeeded: 1881 rows inserted/updated.
[RELIANCE] No data quality issues found in 1881 rows.
[RELIANCE] Pipeline run succeeded: 1881 rows inserted/updated.


## 8. Step 6 — Scheduling

Two ways to schedule this, shown for two different real situations:

**A) The `schedule` library** — runs inside a live Python process. Useful for a quick local demo, but the process has to stay running (this is not real background scheduling, and is not suitable for Colab, which does not stay running unattended).

**B) A real `cron` entry** — the actual, standard way to schedule this on a real always-on machine (a small server, a Raspberry Pi, a cloud VM). This is what you'd actually use in production.


In [9]:
# A) schedule library example - shown as code, NOT run automatically in this notebook
# (running .run_pending() in an infinite loop here would hang the notebook)

scheduling_example_code = '''
import schedule
import time

def scheduled_job():
    conn = sqlite3.connect("market_data.db")
    for label, yf_symbol in TICKERS.items():
        run_pipeline_for_ticker(conn, label, yf_symbol)
    conn.close()

# NSE closes at 3:30 PM IST - schedule the job for shortly after close
schedule.every().day.at("15:45").do(scheduled_job)

while True:
    schedule.run_pending()
    time.sleep(60)
'''
print("Example using the `schedule` library (for a live, always-running process):")
print(scheduling_example_code)


Example using the `schedule` library (for a live, always-running process):

import schedule
import time

def scheduled_job():
    conn = sqlite3.connect("market_data.db")
    for label, yf_symbol in TICKERS.items():
        run_pipeline_for_ticker(conn, label, yf_symbol)
    conn.close()

# NSE closes at 3:30 PM IST - schedule the job for shortly after close
schedule.every().day.at("15:45").do(scheduled_job)

while True:
    schedule.run_pending()
    time.sleep(60)



In [10]:
cron_example = '''
# Real crontab entry (run: crontab -e) for a Linux server, running at 3:45 PM IST every weekday.
# This is the actual production way to schedule this - it does NOT require Colab or any
# Python process to be left running; cron itself wakes up and runs the script.

45 15 * * 1-5 /usr/bin/python3 /home/user/market_data_pipeline.py >> /home/user/pipeline.log 2>&1
'''
print("Real cron entry (for an always-on server, not Colab):")
print(cron_example)


Real cron entry (for an always-on server, not Colab):

# Real crontab entry (run: crontab -e) for a Linux server, running at 3:45 PM IST every weekday.
# This is the actual production way to schedule this - it does NOT require Colab or any
# Python process to be left running; cron itself wakes up and runs the script.

45 15 * * 1-5 /usr/bin/python3 /home/user/market_data_pipeline.py >> /home/user/pipeline.log 2>&1



## 9. Step 7 — Query the Stored Data and Check the Logs

Now that real data is loaded, we can query it directly with SQL — this is the actual point of building a proper database instead of just keeping a DataFrame in memory (which disappears when the notebook session ends).


In [11]:
stored_data = pd.read_sql("SELECT * FROM prices ORDER BY ticker, date", conn)
print(f"Total rows currently stored: {len(stored_data)}")
stored_data.tail(10)


Total rows currently stored: 4408


,ticker,date,open,high,low,close,volume,daily_return
4398,TCS,2026-07-27,2286.000000,2315.000000,2272.399902,2295.600098,4007292,0.018321
4399,TCS,2026-07-28,2325.000000,2417.399902,2323.000000,2398.000000,8248513,0.044607
4400,TCS,2026-07-29,2440.000000,2475.500000,2415.100098,2446.600098,7169419,0.020267
4401,TCS,2026-07-30,2440.000000,2495.000000,2423.000000,2431.800049,4901582,-0.006049
4402,TCS,2026-07-31,2385.000000,2391.000000,2326.100098,2365.600098,4343683,-0.027223
4403,TCS,2026-08-03,2383.899902,2473.699951,2383.000000,2473.699951,3036839,0.045697
4404,TCS,2026-08-04,2463.100098,2467.399902,2435.100098,2460.000000,2792792,-0.005538
4405,TCS,2026-08-05,2460.000000,2460.899902,2408.000000,2413.000000,3014409,-0.019106
4406,TCS,2026-08-06,2417.000000,2434.500000,2368.500000,2373.000000,2729141,-0.016577
4407,TCS,2026-08-07,2375.000000,2457.699951,2375.000000,2452.699951,4547325,NaN


In [12]:
summary_by_ticker = pd.read_sql("""
    SELECT ticker,
           COUNT(*) AS days_stored,
           MIN(date) AS earliest_date,
           MAX(date) AS latest_date,
           ROUND(AVG(daily_return) * 100, 4) AS avg_daily_return_pct
    FROM prices
    GROUP BY ticker
""", conn)
print("Summary per ticker:")
summary_by_ticker


Summary per ticker:


,ticker,days_stored,earliest_date,latest_date,avg_daily_return_pct
0,INFOSYS,1881,2019-01-01,2026-08-07,0.0569
1,RELIANCE,1881,2019-01-01,2026-08-07,0.0679
2,TCS,646,2024-01-01,2026-08-07,-0.0484


In [13]:
pipeline_history = pd.read_sql("SELECT * FROM pipeline_log ORDER BY run_timestamp DESC", conn)
print("Pipeline run history (this is what you'd check to monitor pipeline health over time):")
pipeline_history


Pipeline run history (this is what you'd check to monitor pipeline health over time):


,run_timestamp,ticker,rows_fetched,rows_inserted,status,error_message
0,2026-08-08T20:15:54.367118,RELIANCE,1881,1881,SUCCESS,None
1,2026-08-08T20:15:53.626976,INFOSYS,1881,1881,SUCCESS,None
2,2026-08-08T20:15:52.975497,TCS,1,1,SUCCESS,None


## 10. Honest Conclusion

In [14]:
success_count = (pipeline_history["status"] == "SUCCESS").sum()
failed_count = (pipeline_history["status"] == "FAILED").sum()
print(f"This run: {success_count} ticker(s) succeeded, {failed_count} ticker(s) failed.")
print(f"Total rows currently stored across all tickers: {len(stored_data)}")


This run: 3 ticker(s) succeeded, 0 ticker(s) failed.
Total rows currently stored across all tickers: 4408


### What this project demonstrates, and what it doesn't

**What it demonstrates:** a genuinely reusable ETL pattern — extract, validate, transform, load, with idempotent (safe-to-repeat) writes, incremental fetching instead of full re-downloads every time, and visible logging instead of silent failure. This is a different, complementary skill to the strategy-research projects elsewhere on this CV — it's closer to the data infrastructure work that makes those other projects' data trustworthy in the first place.

**What it doesn't claim:**
1. **Colab is not production hosting.** The SQLite file and any scheduled loop only exist while the Colab session is alive. Real deployment needs an actual always-on machine — this is stated directly rather than implied away.
2. **Single data source, single point of failure.** This relies entirely on `yfinance` / Yahoo Finance. A real production pipeline for anything beyond a personal project would want a paid, SLA-backed data vendor as a fallback, not just error logging.
3. **Three tickers, not a full universe.** This is sized to demonstrate the pattern clearly, not to be a full market data warehouse — the same functions would scale to more tickers with no structural changes needed.
